<a href="https://colab.research.google.com/github/Pedro4Albuquerque/09-DIS-Disney-Dire-o-de-Volume/blob/main/PepilinesDisney.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline 1 | Carregamento dos Dados via API [DIS - Disney]

In [9]:

from typing import List
import requests
import pandas as pd

API_KEY = "YF91FPXACQJ396DA"
SYMBOL = "DIS"

url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={SYMBOL}&apikey={API_KEY}'
r = requests.get(url)
data = r.json()

dados = data["Time Series (Daily)"]

base_dis = pd.DataFrame.from_dict(
    dados,
    orient="index"
)

base_dis.head()

,1. open,2. high,3. low,4. close,5. volume
2026-04-29,100.9400,101.4800,100.6000,101.3000,5533717
2026-04-28,102.6600,103.2899,100.6100,101.4700,6409294
2026-04-27,102.6200,103.8100,102.0600,102.3500,6524285
2026-04-24,103.5950,103.6400,101.9700,102.6000,5974471
2026-04-23,104.9400,105.2000,102.5500,103.6500,6146952


In [10]:
base_dis

,1. open,2. high,3. low,4. close,5. volume
2026-04-29,100.9400,101.4800,100.6000,101.3000,5533717
2026-04-28,102.6600,103.2899,100.6100,101.4700,6409294
2026-04-27,102.6200,103.8100,102.0600,102.3500,6524285
2026-04-24,103.5950,103.6400,101.9700,102.6000,5974471
2026-04-23,104.9400,105.2000,102.5500,103.6500,6146952
...,...,...,...,...,...
2025-12-10,107.1200,109.6650,106.5800,108.8300,11237894
2025-12-09,107.5900,107.7450,106.5500,107.0200,8210364
2025-12-08,105.3000,108.0450,104.8300,107.6300,13177009
2025-12-05,105.1800,106.1700,104.5600,105.3000,10680060


## Blibliotecas de Python

In [11]:
!pip -q install plotly
!pip -q install yellowbrick

In [12]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Pipeline 2 | Organização e Limpeza

In [13]:
base_dis.index = pd.to_datetime(base_dis.index)

base_dis = base_dis.rename(columns={
    "1. open": "Open",
    "2. high": "High",
    "3. low": "Low",
    "4. close": "Close",
    "5. volume": "Volume"
})

base_dis = base_dis.astype(float)

base_dis = base_dis.sort_index()

base_dis.head()

,Open,High,Low,Close,Volume
2025-12-04,105.84,106.215,104.5101,105.47,11471406.0
2025-12-05,105.18,106.170,104.5600,105.30,10680060.0
2025-12-08,105.30,108.045,104.8300,107.63,13177009.0
2025-12-09,107.59,107.745,106.5500,107.02,8210364.0
2025-12-10,107.12,109.665,106.5800,108.83,11237894.0


# Pipeline 3 | Análise Exploratória

In [14]:
base_dis.info()

base_dis.describe()

base_dis.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2025-12-04 to 2026-04-29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    100 non-null    float64
 1   High    100 non-null    float64
 2   Low     100 non-null    float64
 3   Close   100 non-null    float64
 4   Volume  100 non-null    float64
dtypes: float64(5)
memory usage: 4.7 KB


,0
Open,0
High,0
Low,0
Close,0
Volume,0


# Pipeline 4 | Engenharia do Alvo Y

In [15]:
base_dis["fechamento_anterior"] = base_dis["Close"].shift(1)

base_dis["direcao"] = np.where(
    base_dis["Close"] > base_dis["fechamento_anterior"],
    1,
    0
)

base_dis[
    ["Close", "fechamento_anterior", "direcao"]
].head()

,Close,fechamento_anterior,direcao
2025-12-04,105.47,NaN,0
2025-12-05,105.30,105.47,0
2025-12-08,107.63,105.30,1
2025-12-09,107.02,107.63,0
2025-12-10,108.83,107.02,1


# Pipeline 5 | Volume da Véspera

In [16]:

base_dis["volume_anterior"] = base_dis["Volume"].shift(1)

base_dis[
    ["Volume", "volume_anterior", "direcao"]
].head()

,Volume,volume_anterior,direcao
2025-12-04,11471406.0,NaN,0
2025-12-05,10680060.0,11471406.0,0
2025-12-08,13177009.0,10680060.0,1
2025-12-09,8210364.0,13177009.0,0
2025-12-10,11237894.0,8210364.0,1


In [17]:
base_dis

,Open,High,Low,Close,Volume,fechamento_anterior,direcao,volume_anterior
2025-12-04,105.840,106.2150,104.5101,105.47,11471406.0,NaN,0,NaN
2025-12-05,105.180,106.1700,104.5600,105.30,10680060.0,105.47,0,11471406.0
2025-12-08,105.300,108.0450,104.8300,107.63,13177009.0,105.30,1,10680060.0
2025-12-09,107.590,107.7450,106.5500,107.02,8210364.0,107.63,0,13177009.0
2025-12-10,107.120,109.6650,106.5800,108.83,11237894.0,107.02,1,8210364.0
...,...,...,...,...,...,...,...,...
2026-04-23,104.940,105.2000,102.5500,103.65,6146952.0,104.82,0,6886600.0
2026-04-24,103.595,103.6400,101.9700,102.60,5974471.0,103.65,0,6146952.0
2026-04-27,102.620,103.8100,102.0600,102.35,6524285.0,102.60,0,5974471.0
2026-04-28,102.660,103.2899,100.6100,101.47,6409294.0,102.35,0,6524285.0


# Pipeline 6 | Preparação IA

In [18]:
base_dis.dropna(inplace=True)

X_dis = base_dis[["volume_anterior"]].values

Y_dis = base_dis["direcao"].values

# Pipeline 7 | Normalização

In [19]:
from sklearn.preprocessing import StandardScaler

scaler_dis = StandardScaler()

X_dis = scaler_dis.fit_transform(X_dis)

# Pipeline 8 | Treinamento

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_treinamento, X_teste, Y_treinamento, Y_teste = train_test_split(
    X_dis,
    Y_dis,
    test_size=0.15,
    random_state=0,
    shuffle=False
)

modelo_dis = LogisticRegression()

modelo_dis.fit(X_treinamento, Y_treinamento)

LogisticRegression()

# Pipeline 9 | Avaliação

In [21]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

previsoes = modelo_dis.predict(X_teste)

print("Acurácia:")
print(accuracy_score(Y_teste, previsoes))

print("\nMatriz de Confusão:")
print(confusion_matrix(Y_teste, previsoes))

print("\nRelatório:")
print(classification_report(Y_teste, previsoes))

Acurácia:
0.4666666666666667

Matriz de Confusão:
[[7 0]
 [8 0]]

Relatório:
              precision    recall  f1-score   support

           0       0.47      1.00      0.64         7
           1       0.00      0.00      0.00         8

    accuracy                           0.47        15
   macro avg       0.23      0.50      0.32        15
weighted avg       0.22      0.47      0.30        15



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
